# AML SQL-Mode Capability Showcase

Demonstrates every Gremlin step supported in SQL-translation mode (`/query/explain`),
as documented in **Section 17** of `DESIGN_DOCUMENT.md`.

Each section shows the Gremlin traversal, the translated SQL and live results.

**Data model:**
```
Account --[TRANSFER]--> Account
Account --[BELONGS_TO]--> Bank
Account --[FLAGGED_BY]--> Alert
Account --[SENT_VIA]--> Country
Bank    --[LOCATED_IN]--> Country
```

---
## Step 0 — Configuration

Set `BACKEND` then run all cells:

| Value | Mapping | Engine |
|-------|---------|--------|
| `'h2'` | `aml-mapping.json` | H2 embedded (default, no Docker) |
| `'iceberg'` | `iceberg-local-mapping.json` | Trino / Iceberg (Docker required) |

In [11]:
import requests, json as _json, pandas as pd, sys, pathlib
from IPython.display import display, Markdown
from pathlib import Path

# Change to 'iceberg' for the Iceberg/Trino backend
BACKEND = 'iceberg'   # 'h2' | 'iceberg'

# MAX_ROWS controls how many CSV rows are downloaded, normalised, and loaded.
# Increase for a richer graph; decrease for a quick smoke-test.
MAX_ROWS   = 1000000

BASE_URL   = 'http://localhost:7000'
_REPO_ROOT = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                   if (p / 'pom.xml').exists()), Path.cwd())

_MAPPING_FILES = {
    'h2':      'aml-mapping.json',
    'iceberg': 'iceberg-local-mapping.json',
}
MAPPING_PATH = next(
    (str(p) for p in [
        _REPO_ROOT / f"demo/aml/mappings/{_MAPPING_FILES[BACKEND]}",
        Path.cwd()  / f"demo/aml/mappings/{_MAPPING_FILES[BACKEND]}",
    ] if p.exists()), None,
)

print(f'BACKEND      : {BACKEND}')
print(f'MAPPING_PATH : {MAPPING_PATH}')
QUERY_TIMEOUT = 120  # seconds — increase for slow/complex queries

# ── Display options ───────────────────────────────────────────────────────────
# SHOW_PLAN  : True  → show the structured query plan table per cell
#              False → suppress the plan table (cleaner output for demos)
# SQL_LOG    : True  → emit SQL to server logs for every /query/explain call
#              False → suppress SQL from server logs (set False for quiet demos)
#              None  → defer to the server's SQL_TRACE env var (default)
# Note: translated SQL is intentionally only written to server logs ([SQL-TRACE]).
SHOW_PLAN  = True    # True | False
SQL_LOG    = None    # True | False | None

# ── Execution backend ─────────────────────────────────────────────────────────
# GRAPH_PROVIDER is always 'sql' — it is the only supported backend.
# WCOJ (Leapfrog Trie Join) is embedded as an opt-out optimiser inside sql;
# it intercepts multi-hop path traversals and falls back to SQL automatically.
#
# WCOJ tuning env-vars (all optional):
#   WCOJ_ENABLED=true|false         Enable/disable WCOJ (default: true)
#   WCOJ_MAX_EDGES=<n>              In-memory edge limit (default: 5,000,000)
#   WCOJ_DISK_QUOTA_MB=<n>          Per-mapping disk quota in MB (default: 2048)
#   WCOJ_INDEX_TTL_SECONDS=<n>      Index TTL — auto-rebuild stale indices
#                                   (default: 30 s; 0 = disabled / manual only)
#
# Examples:
#   mvn exec:java                                          # WCOJ on, TTL=30 s
#   WCOJ_ENABLED=false mvn exec:java                       # pure SQL, no WCOJ
#   WCOJ_INDEX_TTL_SECONDS=0 mvn exec:java                 # disable TTL
#   WCOJ_MAX_EDGES=500000 mvn exec:java                    # force disk index
#
# The notebook always calls /gremlin/query — the provider is always 'sql'.
# ──────────────────────────────────────────────────────────────────────────────

WCOJ_INDEX_TTL_SECONDS = 300

BACKEND      : iceberg
MAPPING_PATH : /Users/vjoshi/SourceCode/graph-query-engine/demo/aml/mappings/iceberg-local-mapping.json


---
## Step 1 — Helpers (run once)


In [12]:
def explain(gremlin):
    """Call /query/explain.
    Passes ?sql_log= based on the notebook-level SQL_LOG flag:
      SQL_LOG=True  → ?sql_log=true  (force SQL into server logs)
      SQL_LOG=False → ?sql_log=false (suppress SQL from server logs)
      SQL_LOG=None  → no param      (server uses its SQL_TRACE env var default)
    """
    params = {}
    if SQL_LOG is True:  params['sql_log'] = 'true'
    elif SQL_LOG is False: params['sql_log'] = 'false'
    try:
        r = requests.post(f'{BASE_URL}/query/explain', json={'gremlin': gremlin},
                          params=params, timeout=10)
        return r.json() if r.ok else {'error': f'HTTP {r.status_code}: {r.text}'}
    except Exception as e:
        return {'error': str(e)}

def run(gremlin):
    try:
        r = requests.post(f'{BASE_URL}/gremlin/query', json={'gremlin': gremlin}, timeout=QUERY_TIMEOUT)
        return r.json() if r.ok else {'error': f'HTTP {r.status_code}: {r.text}'}
    except Exception as e:
        return {'error': str(e)}

def _active_provider():
    try:
        return requests.get(f'{BASE_URL}/gremlin/provider', timeout=5).json().get('provider', 'sql')
    except Exception:
        return 'sql'

def _fmt_plan(plan):
    """Render a QueryPlan dict as a compact Markdown table."""
    if not plan: return '*No plan available*'
    lines = []
    lines.append('| Field | Value |')
    lines.append('|-------|-------|')
    def row(k, v):
        if v is None: return
        if isinstance(v, list) and len(v) == 0: return
        lines.append(f'| **{k}** | `{v}` |')
    row('rootType',    plan.get('rootType'))
    row('rootLabel',   plan.get('rootLabel'))
    row('rootTable',   plan.get('rootTable'))
    row('dialect',     plan.get('dialect'))
    row('aggregation', plan.get('aggregation'))
    row('simplePath',  plan.get('simplePath'))
    row('limit',       plan.get('limit'))
    row('preHopLimit', plan.get('preHopLimit'))
    row('orderBy',     plan.get('orderBy'))
    row('orderDirection', plan.get('orderDirection'))
    row('dedup',       plan.get('dedup'))
    filters = plan.get('filters') or []
    for f in filters:
        lines.append(f'| **filter** | `{f.get("property")} {f.get("operator","=")} {f.get("value")}` |')
    hops = plan.get('hops') or []
    for i, h in enumerate(hops):
        lines.append(f'| **hop[{i}]** | `{h.get("direction")} → {h.get("labels")}` via `{h.get("resolvedEdgeTable")}` → `{h.get("resolvedTargetTable")}` |')
    projs = plan.get('projections') or []
    for p in projs:
        lines.append(f'| **project** | `{p.get("alias")}` ← `{p.get("kind")}: {p.get("detail")}` |')
    path_by = plan.get('pathByProperties') or []
    if path_by: row('pathBy', ', '.join(path_by))
    where = plan.get('where')
    if where: lines.append(f'| **where** | `{where.get("kind")} {where.get("left","")} {where.get("right","")}` |')
    return '\n'.join(lines)

def show(gremlin, title='', limit=10):
    if title: display(Markdown(f'### {title}'))
    display(Markdown(f'**Gremlin:**\n```groovy\n{gremlin}\n```'))
    ex = explain(gremlin)
    if 'error' in ex:
        display(Markdown(f'**Plan:** *unavailable — {ex["error"]}*'))
        res = run(gremlin)
        if 'error' in res:
            display(Markdown(f'**Error:** {res["error"]}')); return
        rows = res.get('results', [])
        display(Markdown(f'**Results:** {res.get("resultCount", len(rows))}'))
        if rows:
            s = rows[:limit]
            display(pd.DataFrame(s)) if isinstance(s[0], dict) else [print(f'{i}. {v}') for i,v in enumerate(s,1)]
        return

    exec_engine   = ex.get('executionEngine', 'SQL')
    exec_strategy = ex.get('executionStrategy')
    plan          = ex.get('plan')

    # ── Show execution plan / SQL ──────────────────────────────────────────
    if exec_engine == 'WCOJ':
        if SHOW_PLAN:
            display(Markdown(
                f'**Execution engine:** `WCOJ` (Worst-Case Optimal Join — Leapfrog Trie)\n\n'
                f'> {exec_strategy}\n\n'
                f'**Query plan** *(resolved mapping decisions)*:\n\n{_fmt_plan(plan)}'
            ))
        else:
            display(Markdown(f'**Execution engine:** `WCOJ`'))
    elif exec_engine in ('WCOJ_FALLBACK_SQL', 'SQL'):
        display(Markdown(f'**Execution engine:** `{exec_engine}`'))
        if SHOW_PLAN and plan:
            display(Markdown(f'**Query plan:**\n\n{_fmt_plan(plan)}'))
    else:
        # unknown provider (treated as sql)
        display(Markdown(f'**Execution engine:** `{exec_engine}`'))
        if SHOW_PLAN and plan:
            display(Markdown(f'**Query plan:**\n\n{_fmt_plan(plan)}'))

    # ── Execute and show results ───────────────────────────────────────────
    res = run(gremlin)
    if 'error' in res:
        display(Markdown(f'**Error:** {res["error"]}')); return
    rows = res.get('results', [])
    display(Markdown(f'**Results:** {res.get("resultCount", len(rows))}'))
    if rows:
        s = rows[:limit]
        display(pd.DataFrame(s)) if isinstance(s[0], dict) else [print(f'{i}. {v}') for i,v in enumerate(s,1)]

def pick_first_non_empty(candidates):
    for label, g in candidates:
        res = run(g)
        if 'error' in res: continue
        cnt = int(res.get('resultCount', len(res.get('results',[]))))
        if cnt > 0: return label, g, cnt
    label, g = candidates[0]
    res = run(g)
    cnt = int(res.get('resultCount', len(res.get('results',[])))) if 'error' not in res else 0
    return label, g, cnt

def _count(q, default=0):
    try:
        vals = requests.post(f'{BASE_URL}/gremlin/query', json={'gremlin': q}, timeout=10).json().get('results') or []
        return int(vals[0]) if vals else default
    except: return default

print('Helpers ready.')

Helpers ready.


---
## Step 2 — Health check, mapping upload & data load


In [13]:
# Health
try:
    h = requests.get(f'{BASE_URL}/health', timeout=5).json()
    prov = requests.get(f'{BASE_URL}/gremlin/provider', timeout=5).json().get('provider','?')
    display(Markdown(f'**Health:** `{h["status"]}` — provider: **`{prov}`** — backends: `{[b["id"] for b in h.get("backends",[])]}`'))
except Exception as e:
    display(Markdown(f'**Backend not reachable:** {e}'))

_MAPPING_IDS = {
    'h2':      'aml-h2',
    'iceberg': 'aml-iceberg',
}

# ── Iceberg container guard (iceberg mode only) ────────────────────────
if BACKEND == 'iceberg':
    import subprocess as _sp
    def _containers_running():
        try:
            r = _sp.run(['docker', 'ps', '--format', '{{.Names}}'],
                        capture_output=True, text=True, timeout=10)
            return 'iceberg-trino' in r.stdout and 'iceberg-minio' in r.stdout
        except Exception:
            return False
    if not _containers_running():
        display(Markdown(
            '\u274c **Iceberg Docker containers are not running.**\n\n'
            'Start them first:\n```bash\nbash demo/infra/scripts/iceberg_local_up.sh\n```'
        ))
    else:
        display(Markdown('\u2705 Iceberg containers are running'))

# ── Upload mapping ─────────────────────────────────────────────────────
if MAPPING_PATH:
    with open(MAPPING_PATH, 'rb') as f:
        resp = requests.post(
            f'{BASE_URL}/mapping/upload',
            files={'file': (pathlib.Path(MAPPING_PATH).name, f, 'application/json')},
            params={'id': _MAPPING_IDS[BACKEND], 'name': f'AML {BACKEND.title()}', 'activate': 'true'},
            timeout=15,
        )
    if resp.ok:
        d = resp.json()
        msg = f'\u2705 Mapping uploaded \u2014 `{d["mappingId"]}`'
        if d.get('autoRegisteredBackends'):
            msg += f'  *(auto-registered backends: `{d["autoRegisteredBackends"]}`)*'
        display(Markdown(msg))
    else:
        display(Markdown(f'\u274c Mapping upload failed: `{resp.text[:200]}`'))
else:
    display(Markdown(f'**Mapping not found** \u2014 expected `demo/aml/mappings/{_MAPPING_FILES[BACKEND]}`'))

# ── Data load ──────────────────────────────────────────────────────────
# prepare_aml_data.py handles the full pipeline end-to-end:
#   1. Download raw CSV from Kaggle (skips if already present)
#   2. Normalise to engine format writing exactly MAX_ROWS rows
#   3. Bulk-load into H2 (CSVREAD) or Iceberg (MinIO upload + INSERT SELECT)
#
# MAX_ROWS controls ALL three stages consistently.
# Pass --force-download to re-download; --wipe to clear existing DB data.
import subprocess as _sp
_prepare = _REPO_ROOT / 'demo' / 'aml' / 'scripts' / 'prepare_aml_data.py'

ac = _count("g.V().hasLabel('Account').count()")
# Threshold: for N transaction rows, HI-Small produces ~N*0.14 unique accounts.
# Re-run the prepare script if we have fewer accounts than 10% of MAX_ROWS
# (i.e., the existing load used a smaller MAX_ROWS setting).
_reload_threshold = max(1, int(MAX_ROWS * 0.10))
if ac >= _reload_threshold:
    display(Markdown(f'\u2713 **Data already loaded** \u2014 {ac:,} accounts in graph '
                     f'(threshold for MAX_ROWS={MAX_ROWS:,} is {_reload_threshold:,}). '
                     f'Re-run with `--wipe` in the prepare script to force reload.'))
elif not _prepare.exists():
    display(Markdown(f'\u274c **Prepare script not found:** `{_prepare}`'))
else:
    display(Markdown(
        f'**Preparing AML data** (`MAX_ROWS={MAX_ROWS:,}`, backend=`{BACKEND}`) ...\n\n'
        f'Pipeline: **Kaggle download** \u2192 **normalise** \u2192 **CSV bulk load** into {BACKEND.upper()}\n\n'
        f'> First run downloads from Kaggle (~30 MB for HI-Small). Subsequent runs skip download + normalise.'
    ))
    # Run with inherited stdout/stderr so output streams live to the notebook.
    # This ensures Kaggle download progress and seed errors are always visible.
    result = _sp.run(
        [sys.executable, str(_prepare),
         '--rows',    str(MAX_ROWS),
         '--backend', BACKEND],
        cwd=str(_REPO_ROOT)
    )
    if result.returncode != 0:
        display(Markdown(
            f'\u2717 **Prepare failed** (exit {result.returncode}) \u2014 '
            f'see output above for details.'
        ))
    else:
        ac = _count("g.V().hasLabel('Account').count()")
        display(Markdown(f'\u2705 **Ready** \u2014 {ac:,} accounts in graph'))


**Health:** `ok` — provider: **`sql-multi`** — backends: `['default', 'iceberg']`

✅ Iceberg containers are running

✅ Mapping uploaded — `aml-iceberg`  *(auto-registered backends: `['iceberg']`)*

✓ **Data already loaded** — 423,684 accounts in graph (threshold for MAX_ROWS=1,000,000 is 100,000). Re-run with `--wipe` in the prepare script to force reload.

---
## 1 — Root steps: `g.V()`, `g.E()`, `g.V(id)`


In [4]:
show("g.V().hasLabel('Account').count()", title='1a  g.V() — count all Account vertices')


### 1a  g.V() — count all Account vertices

**Gremlin:**
```groovy
g.V().hasLabel('Account').count()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `count` |

**Results:** 1

1. 423684


In [5]:
show("g.E().hasLabel('TRANSFER').count()", title='1b  g.E() — count all TRANSFER edges')


### 1b  g.E() — count all TRANSFER edges

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').count()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `count` |

**Results:** 1

1. 1000000


In [6]:
_fid = (run("g.V().hasLabel('Account').values('accountId').limit(1)").get('results') or ['ACC-001'])[0]
show(f"g.V().hasLabel('Account').has('accountId','{_fid}').project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore')",
     title='1c  g.V(id) — look up a specific account by property')


### 1c  g.V(id) — look up a specific account by property

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('accountId','8000EBD30').project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **filter** | `accountId = 8000EBD30` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `riskScore` ← `EDGE_PROPERTY: riskScore` |

**Results:** 1

,accountId,bankId,riskScore
0,[8000EBD30],[010],[0.59]


---
## 2 — Filter steps: `has`, `hasLabel`, `hasId`, `hasNot`, `is`


In [7]:
show("g.V().hasLabel('Account').has('isBlocked','true').project('accountId','bankId','isBlocked').by('accountId').by('bankId').by('isBlocked').limit(10)", title='2a  has(k,v) — blocked accounts')


### 2a  has(k,v) — blocked accounts

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('isBlocked','true').project('accountId','bankId','isBlocked').by('accountId').by('bankId').by('isBlocked').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **filter** | `isBlocked = true` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `isBlocked` ← `EDGE_PROPERTY: isBlocked` |

**Results:** 10

,accountId,bankId,isBlocked
0,[8017BF800],[002439],[true]
1,[80AEF5310],[0211050],[true]
2,[8005F0B50],[010],[true]
3,[8006AF500],[010],[true]
4,[8016BBF40],[010],[true]
5,[80D454CA0],[0222363],[true]
6,[800117590],[012],[true]
7,[80014B530],[010],[true]
8,[800156890],[010],[true]
9,[800145A20],[00220],[true]


In [8]:
show("g.V().hasLabel('Account').has('riskScore', gt(0.7)).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').order().by(select('riskScore'), Order.desc).limit(10)", title='2b  has(k, gt(v)) — high risk-score accounts (> 0.7)')


### 2b  has(k, gt(v)) — high risk-score accounts (> 0.7)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('riskScore', gt(0.7)).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').order().by(select('riskScore'), Order.desc).limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **orderBy** | `riskScore` |
| **orderDirection** | `DESC` |
| **filter** | `riskScore > 0.7` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `riskScore` ← `EDGE_PROPERTY: riskScore` |

**Results:** 10

,accountId,bankId,riskScore
0,[800467C10],[001],[0.99]
1,[800321170],[01362],[0.99]
2,[8008E6E00],[001124],[0.99]
3,[80032DA20],[01588],[0.99]
4,[8003ED1F0],[001],[0.99]
5,[8002B5CF0],[03401],[0.99]
6,[800A5F910],[031794],[0.99]
7,[80067E330],[01601],[0.99]
8,[8009DAD60],[03760],[0.99]
9,[800462C10],[03212],[0.99]


In [9]:
_vid = (run("g.V().hasLabel('Account').id().limit(1)").get('results') or [1])[0]
_vlit = str(_vid) if isinstance(_vid,(int,float)) or (isinstance(_vid,str) and _vid.isdigit()) else f"'{_vid}'"
show(f"g.V().hasLabel('Account').hasId({_vlit}).project('accountId','bankId').by('accountId').by('bankId')",
     title='2c  hasId(id) — look up vertex by graph ID')


### 2c  hasId(id) — look up vertex by graph ID

**Gremlin:**
```groovy
g.V().hasLabel('Account').hasId(1).project('accountId','bankId').by('accountId').by('bankId')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **filter** | `id = 1` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 1

,accountId,bankId
0,[8000EBD30],[010]


In [10]:
_no_date_count = _count("g.V().hasLabel('Account').hasNot('openedDate').count()")
if _no_date_count == 0:
    display(Markdown(
        '> ⚠️ **`openedDate` has no NULLs in the current dataset.** '
        'This means the Iceberg table was loaded by an older version of the seeder.\n\n'
        '> Re-run **Step 2** — the seeder now auto-detects this and reloads the accounts table '
        'so that ~30% of accounts have `NULL` for `openedDate`.'
    ))
else:
    show("g.V().hasLabel('Account').hasNot('openedDate').project('accountId','bankId').by('accountId').by('bankId').limit(10)",
         title='2d  hasNot(k) — accounts with no openedDate (IS NULL)')


### 2d  hasNot(k) — accounts with no openedDate (IS NULL)

**Gremlin:**
```groovy
g.V().hasLabel('Account').hasNot('openedDate').project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **filter** | `openedDate IS NULL None` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 10

,accountId,bankId
0,[8000F5030],[012]
1,[8000EC1E0],[001]
2,[80AEF5310],[0211050]
3,[8011305D0],[011813]
4,[8000F47F0],[001]
5,[8000F4FE0],[001]
6,[805B34210],[013037]
7,[8005DFEB0],[01420]
8,[8005E18F0],[01665]
9,[8005E24C0],[01665]


In [11]:
show("g.V().hasLabel('Account').values('riskScore').is(gt(0.8)).limit(10)",
     title='2e  values(p).is(pred) — risk scores above 0.8')


### 2e  values(p).is(pred) — risk scores above 0.8

**Gremlin:**
```groovy
g.V().hasLabel('Account').values('riskScore').is(gt(0.8)).limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **filter** | `riskScore > 0.8` |

**Results:** 10

1. 0.94
2. 0.92
3. 0.91
4. 0.91
5. 0.84
6. 0.87
7. 0.83
8. 0.93
9. 0.83
10. 0.84


---
## 3 — Hop steps: `out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV`


In [12]:
show("g.V().hasLabel('Account').limit(5).out('BELONGS_TO').project('bankId','bankName').by('bankId').by('bankName')", title='3a  out(l) — Account → Bank')


### 3a  out(l) — Account → Bank

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).out('BELONGS_TO').project('bankId','bankName').by('bankId').by('bankName')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **hop[0]** | `out → ['BELONGS_TO']` via `iceberg.aml.account_bank` → `iceberg.aml.banks` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `bankName` ← `EDGE_PROPERTY: bankName` |

**Results:** 5

,bankId,bankName
0,[010],[Bank-010]
1,[03208],[Bank-03208]
2,[001],[Bank-001]
3,[03209],[Bank-03209]
4,[012],[Bank-012]


In [13]:
show("g.V().hasLabel('Bank').limit(5).in('BELONGS_TO').project('accountId','bankId').by('accountId').by('bankId')", title='3b  in(l) — Bank ← Account')


### 3b  in(l) — Bank ← Account

**Gremlin:**
```groovy
g.V().hasLabel('Bank').limit(5).in('BELONGS_TO').project('accountId','bankId').by('accountId').by('bankId')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Bank` |
| **rootTable** | `iceberg.aml.banks` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **hop[0]** | `in → ['BELONGS_TO']` via `iceberg.aml.account_bank` → `iceberg.aml.accounts` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 6703

,accountId,bankId
0,[8000EBD30],[010]
1,[8000F4580],[03208]
2,[8000F5340],[001]
3,[8000F4670],[03209]
4,[8000F5030],[012]
5,[8000F5200],[010]
6,[8000F5AD0],[001]
7,[8000EBAC0],[001]
8,[8000EC1E0],[001]
9,[8000EC280],[012]


In [14]:
show("g.V().hasLabel('Account').limit(3).both('TRANSFER').dedup().project('accountId','bankId').by('accountId').by('bankId').limit(10)", title='3c  both(l) — bidirectional TRANSFER neighbours')


### 3c  both(l) — bidirectional TRANSFER neighbours

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(3).both('TRANSFER').dedup().project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **limit** | `10` |
| **preHopLimit** | `3` |
| **dedup** | `True` |
| **hop[0]** | `both → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 7

,accountId,bankId
0,[8000F4580],[03208]
1,[804D2AB40],[011974]
2,[8000EBD30],[010]
3,[809FCF310],[006129]
4,[80B22DCB0],[004726]
5,[8000F5340],[001]
6,[81422D380],[009482]


In [15]:
show("g.V().hasLabel('Account').limit(5).outE('TRANSFER').project('transactionId','amount','currency','isLaundering').by('transactionId').by('amount').by('currency').by('isLaundering').limit(10)", title='3d  outE(l) — outgoing TRANSFER edges')


### 3d  outE(l) — outgoing TRANSFER edges

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).outE('TRANSFER').project('transactionId','amount','currency','isLaundering').by('transactionId').by('amount').by('currency').by('isLaundering').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **limit** | `10` |
| **preHopLimit** | `5` |
| **hop[0]** | `outE → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **project** | `transactionId` ← `EDGE_PROPERTY: transactionId` |
| **project** | `amount` ← `EDGE_PROPERTY: amount` |
| **project** | `currency` ← `EDGE_PROPERTY: currency` |
| **project** | `isLaundering` ← `EDGE_PROPERTY: isLaundering` |

**Results:** 9

,transactionId,amount,currency,isLaundering
0,[1],[3697.34],[US Dollar],[0]
1,[2],[0.01],[US Dollar],[0]
2,[3],[14675.57],[US Dollar],[0]
3,[4],[2806.97],[US Dollar],[0]
4,[947462],[23.54],[US Dollar],[0]
5,[280626],[109192.02],[US Dollar],[0]
6,[280627],[616552.72],[Brazil Real],[0]
7,[982879],[77.4],[US Dollar],[0]
8,[982880],[18.69],[US Dollar],[0]


In [16]:
show("g.V().hasLabel('Account').limit(5).inE('TRANSFER').project('transactionId','amount','currency').by('transactionId').by('amount').by('currency').limit(10)", title='3e  inE(l) — incoming TRANSFER edges')


### 3e  inE(l) — incoming TRANSFER edges

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).inE('TRANSFER').project('transactionId','amount','currency').by('transactionId').by('amount').by('currency').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **limit** | `10` |
| **preHopLimit** | `5` |
| **hop[0]** | `inE → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **project** | `transactionId` ← `EDGE_PROPERTY: transactionId` |
| **project** | `amount` ← `EDGE_PROPERTY: amount` |
| **project** | `currency` ← `EDGE_PROPERTY: currency` |

**Results:** 10

,transactionId,amount,currency
0,[1],[3697.34],[US Dollar]
1,[2],[0.01],[US Dollar]
2,[3],[14675.57],[US Dollar]
3,[4],[2806.97],[US Dollar]
4,[48203],[18.66],[US Dollar]
5,[48207],[8.68],[US Dollar]
6,[63279],[70.26],[US Dollar]
7,[68689],[141.06],[US Dollar]
8,[120954],[301.65],[US Dollar]
9,[947462],[23.54],[US Dollar]


In [17]:
show("g.V().hasLabel('Account').limit(3).bothE('TRANSFER').project('transactionId','amount').by('transactionId').by('amount').limit(10)", title='3f  bothE(l) — all TRANSFER edges')


### 3f  bothE(l) — all TRANSFER edges

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(3).bothE('TRANSFER').project('transactionId','amount').by('transactionId').by('amount').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **limit** | `10` |
| **preHopLimit** | `3` |
| **hop[0]** | `bothE → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **project** | `transactionId` ← `EDGE_PROPERTY: transactionId` |
| **project** | `amount` ← `EDGE_PROPERTY: amount` |

**Results:** 8

,transactionId,amount
0,[1],[3697.34]
1,[2],[0.01]
2,[2],[0.01]
3,[63279],[70.26]
4,[68689],[141.06]
5,[120954],[301.65]
6,[982879],[77.4]
7,[982880],[18.69]


In [18]:
show("g.E().hasLabel('TRANSFER').limit(5).outV().project('accountId','bankId').by('accountId').by('bankId')", title='3g  outV() — source vertex of TRANSFER')


### 3g  outV() — source vertex of TRANSFER

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).outV().project('accountId','bankId').by('accountId').by('bankId')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **hop[0]** | `outV → []` via `None` → `None` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 5

,accountId,bankId
0,[8000EBD30],[010]
1,[8000F4580],[03208]
2,[8000F5340],[001]
3,[8000F4670],[03209]
4,[8000F5030],[012]


In [19]:
show("g.E().hasLabel('TRANSFER').limit(5).inV().project('accountId','bankId').by('accountId').by('bankId')", title='3h  inV() — destination vertex of TRANSFER')


### 3h  inV() — destination vertex of TRANSFER

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).inV().project('accountId','bankId').by('accountId').by('bankId')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **hop[0]** | `inV → []` via `None` → `None` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 5

,accountId,bankId
0,[8000EBD30],[010]
1,[8000F5340],[001]
2,[8000F4670],[03209]
3,[8000F5030],[012]
4,[8000F5200],[010]


In [20]:
show("g.E().hasLabel('TRANSFER').limit(5).bothV().dedup().project('accountId','bankId').by('accountId').by('bankId')", title='3i  bothV() — all endpoint vertices of TRANSFER')


### 3i  bothV() — all endpoint vertices of TRANSFER

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).bothV().dedup().project('accountId','bankId').by('accountId').by('bankId')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **dedup** | `True` |
| **hop[0]** | `bothV → []` via `None` → `None` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 5

,accountId,bankId
0,[8000EBD30],[010]
1,[8000F4580],[03208]
2,[8000F5340],[001]
3,[8000F4670],[03209]
4,[8000F5030],[012]


In [21]:
show("g.V().hasLabel('Account').limit(5).outE('BELONGS_TO').otherV().project('bankId','bankName').by('bankId').by('bankName')", title='3j  otherV() — opposite endpoint after outE')


### 3j  otherV() — opposite endpoint after outE

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).outE('BELONGS_TO').otherV().project('bankId','bankName').by('bankId').by('bankName')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **hop[0]** | `out → ['BELONGS_TO']` via `iceberg.aml.account_bank` → `iceberg.aml.banks` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `bankName` ← `EDGE_PROPERTY: bankName` |

**Results:** 5

,bankId,bankName
0,[010],[Bank-010]
1,[03208],[Bank-03208]
2,[001],[Bank-001]
3,[03209],[Bank-03209]
4,[012],[Bank-012]


---
## 4 — Repeat / path steps


In [22]:
show("g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(2).path().by('accountId').limit(10)", title='4a  repeat().times(n) + simplePath() — two-hop money trail')


### 4a  repeat().times(n) + simplePath() — two-hop money trail

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(2).path().by('accountId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **simplePath** | `True` |
| **limit** | `10` |
| **preHopLimit** | `1` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[1]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **pathBy** | `accountId` |
| **where** | `EDGE_EXISTS outE TRANSFER` |

**Results:** 0

In [23]:
show("g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)", title='4b  repeat multi-label + path().by(p) — Account → Bank → Country')


### 4b  repeat multi-label + path().by(p) — Account → Bank → Country

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **simplePath** | `True` |
| **limit** | `10` |
| **preHopLimit** | `1` |
| **hop[0]** | `out → ['BELONGS_TO']` via `iceberg.aml.account_bank` → `iceberg.aml.banks` |
| **hop[1]** | `out → ['LOCATED_IN']` via `iceberg.aml.bank_country` → `iceberg.aml.countries` |
| **pathBy** | `accountId, bankName, countryName` |

**Results:** 1

1. ['8000EBD30', 'Bank-010', 'Singapore']


---
## 5 — Aggregation: `count`, `sum`, `mean`, `groupCount`


In [24]:
show("g.V().hasLabel('Account').count()", title='5a  count()')


### 5a  count()

**Gremlin:**
```groovy
g.V().hasLabel('Account').count()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `count` |

**Results:** 1

1. 423684


In [25]:
show("g.V().hasLabel('Account').dedup().count()", title='5b  dedup().count() — COUNT DISTINCT')


### 5b  dedup().count() — COUNT DISTINCT

**Gremlin:**
```groovy
g.V().hasLabel('Account').dedup().count()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `count` |
| **dedup** | `True` |

**Results:** 1

1. 423684


In [26]:
show("g.E().hasLabel('TRANSFER').values('amount').sum()", title='5c  values(amount).sum()')


### 5c  values(amount).sum()

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').values('amount').sum()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `sum` |

**Results:** 1

1. 6044978789025.301


In [27]:
show("g.E().hasLabel('TRANSFER').values('amount').mean()", title='5d  values(amount).mean()')


### 5d  values(amount).mean()

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').values('amount').mean()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `mean` |

**Results:** 1

1. 6044978.789025301


In [28]:
show("g.V().hasLabel('Account').groupCount().by('bankId').order().by(values, desc).limit(10)", title='5e  groupCount().by(p) — accounts per bank')


### 5e  groupCount().by(p) — accounts per bank

**Gremlin:**
```groovy
g.V().hasLabel('Account').groupCount().by('bankId').order().by(values, desc).limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `groupCount` |
| **preHopLimit** | `10` |
| **orderDirection** | `DESC` |

**Results:** 10

,bankId,count
0,[012],[2499]
1,[010],[2092]
2,[001],[2082]
3,[003],[1704]
4,[015],[1583]
5,[007],[1583]
6,[020],[1546]
7,[028],[1476]
8,[0211],[1432]
9,[0116],[1430]


---
## 6 — Projection: `values`, `project().by()`, `valueMap`, `by(identity())`


In [29]:
show("g.V().hasLabel('Account').values('accountId').limit(10)", title='6a  values(p) — single property column')


### 6a  values(p) — single property column

**Gremlin:**
```groovy
g.V().hasLabel('Account').values('accountId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |

**Results:** 10

1. 8000EBD30
2. 8000F4580
3. 8000F5340
4. 8000F4670
5. 8000F5030
6. 8000F5200
7. 8000F5AD0
8. 8000EBAC0
9. 8000EC1E0
10. 8000EC280


In [30]:
show("g.V().hasLabel('Account').project('accountId','bankId','riskScore','accountType').by('accountId').by('bankId').by('riskScore').by('accountType').limit(10)", title='6b  project().by(p) — multi-column SELECT')


### 6b  project().by(p) — multi-column SELECT

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','bankId','riskScore','accountType').by('accountId').by('bankId').by('riskScore').by('accountType').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `riskScore` ← `EDGE_PROPERTY: riskScore` |
| **project** | `accountType` ← `EDGE_PROPERTY: accountType` |

**Results:** 10

,accountId,bankId,riskScore,accountType
0,[8000EBD30],[010],[0.59],[BUSINESS]
1,[8000F4670],[03209],[0.26],[PERSONAL]
2,[8000F5030],[012],[0.58],[BUSINESS]
3,[8000F5200],[010],[0.42],[BUSINESS]
4,[8000F5AD0],[001],[0.16],[PERSONAL]
5,[8000F4580],[03208],[0.71],[PERSONAL]
6,[8000F5340],[001],[0.41],[PERSONAL]
7,[8000EBAC0],[001],[0.62],[PERSONAL]
8,[8000EC1E0],[001],[0.44],[BUSINESS]
9,[8000EC280],[012],[0.28],[PERSONAL]


In [31]:
show("g.V().hasLabel('Account').valueMap().limit(5)", title='6c  valueMap() — all properties (vertex)')


### 6c  valueMap() — all properties (vertex)

**Gremlin:**
```groovy
g.V().hasLabel('Account').valueMap().limit(5)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |

**Results:** 5

,accountId,bankId,riskScore,isBlocked,openedDate,accountType
0,[8000EBD30],[010],[0.59],[false],[2022-09-01],[BUSINESS]
1,[8000F4580],[03208],[0.71],[false],[2022-09-01],[PERSONAL]
2,[8000F5340],[001],[0.41],[false],[2022-09-01],[PERSONAL]
3,[8000F4670],[03209],[0.26],[false],[2022-09-01],[PERSONAL]
4,[8000F5030],[012],[0.58],[false],None,[BUSINESS]


In [32]:
show("g.E().hasLabel('TRANSFER').valueMap().limit(5)", title='6d  valueMap() — all properties (edge)')


### 6d  valueMap() — all properties (edge)

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').valueMap().limit(5)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |

**Results:** 5

,transactionId,amount,currency,paymentFormat,eventTime,isLaundering
0,[726901],[18007.7],[Mexican Peso],[Cheque],[2022/09/01 12:14],[0]
1,[726902],[890.79],[Mexican Peso],[ACH],[2022/09/01 12:06],[0]
2,[726903],[1411.26],[Mexican Peso],[Wire],[2022/09/01 12:20],[0]
3,[726904],[57365.74],[Mexican Peso],[Cheque],[2022/09/01 12:15],[0]
4,[726905],[63082.38],[Mexican Peso],[Cheque],[2022/09/01 12:13],[0]


In [33]:
show("g.V().hasLabel('Account').project('id','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(10)", title='6e  by(identity()) — emit element id')


### 6e  by(identity()) — emit element id

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('id','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `id` ← `IDENTITY: id` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 10

,id,accountId,bankId
0,[1],[8000EBD30],[010]
1,[2],[8000F4580],[03208]
2,[3],[8000F5340],[001]
3,[4],[8000F4670],[03209]
4,[5],[8000F5030],[012]
5,[6],[8000F5200],[010]
6,[7],[8000F5AD0],[001]
7,[8],[8000EBAC0],[001]
8,[9],[8000EC1E0],[001]
9,[10],[8000EC280],[012]


In [34]:
show("g.V().hasLabel('Account').limit(10).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())", title='6f  by(out(l).values(p).fold()) — neighbour property list')


### 6f  by(out(l).values(p).fold()) — neighbour property list

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(10).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `bankName` ← `OUT_NEIGHBOR_PROPERTY: out:BELONGS_TO:bankName` |

**Results:** 10

,accountId,bankId,bankName
0,[8000F4670],[03209],[Bank-03209]
1,[8000F5030],[012],[Bank-012]
2,[8000F5200],[010],[Bank-010]
3,[8000F5AD0],[001],[Bank-001]
4,[8000F4580],[03208],[Bank-03208]
5,[8000F5340],[001],[Bank-001]
6,[8000EBAC0],[001],[Bank-001]
7,[8000EC1E0],[001],[Bank-001]
8,[8000EC280],[012],[Bank-012]
9,[8000EBD30],[010],[Bank-010]


In [35]:
show("g.V().hasLabel('Account').project('accountId','outDegree','inDegree').by('accountId').by(outE('TRANSFER').count()).by(inE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)", title='6g  by(outE/inE.count()) — degree subqueries')


### 6g  by(outE/inE.count()) — degree subqueries

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','outDegree','inDegree').by('accountId').by(outE('TRANSFER').count()).by(inE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **orderBy** | `outDegree` |
| **orderDirection** | `DESC` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `outDegree` ← `EDGE_DEGREE: out:TRANSFER` |
| **project** | `inDegree` ← `EDGE_DEGREE: in:TRANSFER` |

**Results:** 10

,accountId,outDegree,inDegree
0,[100428660],[19968],[5]
1,[1004286A8],[11967],[3]
2,[100428978],[2441],[0]
3,[1004286F0],[2142],[2]
4,[1004289C0],[1993],[4]
5,[100428810],[1982],[2]
6,[100428780],[1968],[5]
7,[1004287C8],[1596],[9]
8,[100428738],[1578],[7]
9,[100428A51],[1522],[0]


In [36]:
show("g.V().hasLabel('Account').project('accountId','riskBucket').by('accountId').by(choose(values('riskScore').is(gte(0.7)), constant('HIGH'), constant('NORMAL'))).limit(15)", title='6h  by(choose(...)) — CASE WHEN projection')


### 6h  by(choose(...)) — CASE WHEN projection

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','riskBucket').by('accountId').by(choose(values('riskScore').is(gte(0.7)), constant('HIGH'), constant('NORMAL'))).limit(15)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `15` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `riskBucket` ← `CHOOSE_VALUES_IS_CONSTANT: choose(values('riskScore').is(gte(0.7)), constant('HIGH'), constant('NORMAL'))` |

**Results:** 15

,accountId,riskBucket
0,[8000EBD30],[NORMAL]
1,[8000F4670],[NORMAL]
2,[8000F5030],[NORMAL]
3,[8000F5200],[NORMAL]
4,[8000F5AD0],[NORMAL]
5,[8000F4580],[HIGH]
6,[8000F5340],[NORMAL]
7,[8000EBAC0],[NORMAL]
8,[8000EC1E0],[NORMAL]
9,[8000EC280],[NORMAL]


In [37]:
show("g.E().hasLabel('TRANSFER').limit(5).project('fromAccountId','toAccountId','amount').by(outV().values('accountId')).by(inV().values('accountId')).by('amount')", title='6i  by(outV/inV.values(p)) — edge project with endpoints')


### 6i  by(outV/inV.values(p)) — edge project with endpoints

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).project('fromAccountId','toAccountId','amount').by(outV().values('accountId')).by(inV().values('accountId')).by('amount')
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **project** | `fromAccountId` ← `OUT_VERTEX_PROPERTY: accountId` |
| **project** | `toAccountId` ← `IN_VERTEX_PROPERTY: accountId` |
| **project** | `amount` ← `EDGE_PROPERTY: amount` |

**Results:** 5

,fromAccountId,toAccountId,amount
0,[80BD32290],[80BEEF330],[18007.7]
1,[80BD32290],[80BEEF330],[890.79]
2,[80BD32290],[80BEEF330],[1411.26]
3,[80B7B4990],[80BEF0110],[57365.74]
4,[80A4377F0],[80BF11D00],[63082.38]


---
## 7 — Ordering and paging: `order().by`, `limit`, `dedup`


In [38]:
show("g.V().hasLabel('Account').order().by('riskScore', Order.desc).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)", title='7a  order().by(p, desc) — top-risk accounts')


### 7a  order().by(p, desc) — top-risk accounts

**Gremlin:**
```groovy
g.V().hasLabel('Account').order().by('riskScore', Order.desc).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **orderBy** | `riskScore` |
| **orderDirection** | `DESC` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `riskScore` ← `EDGE_PROPERTY: riskScore` |

**Results:** 10

,accountId,bankId,riskScore
0,[800462C10],[03212],[0.99]
1,[80067E330],[01601],[0.99]
2,[8008E6E00],[001124],[0.99]
3,[800321170],[01362],[0.99]
4,[8002B5CF0],[03401],[0.99]
5,[8003ED1F0],[001],[0.99]
6,[800A5F910],[031794],[0.99]
7,[800467C10],[001],[0.99]
8,[8009DAD60],[03760],[0.99]
9,[80032DA20],[01588],[0.99]


In [39]:
show("g.V().hasLabel('Account').project('accountId','outDegree').by('accountId').by(outE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)", title='7b  order().by(select(a), desc) — order by projected alias')


### 7b  order().by(select(a), desc) — order by projected alias

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','outDegree').by('accountId').by(outE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **orderBy** | `outDegree` |
| **orderDirection** | `DESC` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `outDegree` ← `EDGE_DEGREE: out:TRANSFER` |

**Results:** 10

,accountId,outDegree
0,[100428660],[19968]
1,[1004286A8],[11967]
2,[100428978],[2441]
3,[1004286F0],[2142]
4,[1004289C0],[1993]
5,[100428810],[1982]
6,[100428780],[1968]
7,[1004287C8],[1596]
8,[100428738],[1578]
9,[100428A51],[1522]


In [40]:
show("g.V().hasLabel('Account').dedup().count()", title='7c  dedup() — COUNT DISTINCT')


### 7c  dedup() — COUNT DISTINCT

**Gremlin:**
```groovy
g.V().hasLabel('Account').dedup().count()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `count` |
| **dedup** | `True` |

**Results:** 1

1. 423684


---
## 8 — Alias / select / where


In [41]:
show("g.V().hasLabel('Account').as('a').out('TRANSFER').as('b').where('a', neq('b')).select('a','b').by('accountId').by('accountId').limit(10)", title='8a  as + where(neq) — self-loop exclusion')


### 8a  as + where(neq) — self-loop exclusion

**Gremlin:**
```groovy
g.V().hasLabel('Account').as('a').out('TRANSFER').as('b').where('a', neq('b')).select('a','b').by('accountId').by('accountId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **limit** | `10` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **where** | `NEQ_ALIAS a b` |

**Results:** 10

,id,account_id,bank_id,risk_score,is_blocked,opened_date,account_type
0,[3],[8000F5340],[001],[0.41],[false],[2022-09-01],[PERSONAL]
1,[3],[8000F5340],[001],[0.41],[false],[2022-09-01],[PERSONAL]
2,[3],[8000F5340],[001],[0.41],[false],[2022-09-01],[PERSONAL]
3,[3],[8000F5340],[001],[0.41],[false],[2022-09-01],[PERSONAL]
4,[5],[8000F5030],[012],[0.58],[false],None,[BUSINESS]
5,[5],[8000F5030],[012],[0.58],[false],None,[BUSINESS]
6,[7],[8000F5AD0],[001],[0.16],[false],[2022-09-01],[PERSONAL]
7,[9],[8000EC1E0],[001],[0.44],[false],None,[BUSINESS]
8,[11],[8017BF800],[002439],[0.94],[true],[2022-09-01],[BUSINESS]
9,[13],[80AEF5310],[0211050],[0.92],[true],None,[PERSONAL]


In [42]:
show("g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)", title='8b  where(outE.has) — correlated EXISTS')


### 8b  where(outE.has) — correlated EXISTS

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **where** | `EDGE_EXISTS outE TRANSFER` |

**Results:** 10

,accountId,bankId
0,[801009860],[01674]
1,[800054B70],[012]
2,[800056160],[001]
3,[100428660],[070]
4,[800117590],[012]
5,[800119810],[010]
6,[800137450],[012]
7,[8001A6140],[010]
8,[8001BB380],[010]
9,[8002B9DE0],[01420]


In [43]:
show("g.V().hasLabel('Account').where(inE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)", title='8c  where(inE.has) — EXISTS on incoming edge')


### 8c  where(inE.has) — EXISTS on incoming edge

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(inE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **where** | `EDGE_EXISTS inE TRANSFER` |

**Results:** 10

,accountId,bankId
0,[800054B70],[012]
1,[800115860],[00220]
2,[800119810],[010]
3,[800164250],[010]
4,[80016DA30],[010]
5,[8001A6140],[010]
6,[8001ECD70],[01362]
7,[8002B9DE0],[01420]
8,[8004030A0],[01292]
9,[800449990],[010]


In [44]:
show("g.V().hasLabel('Account').where(bothE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)", title='8d  where(bothE.has) — EXISTS either direction')


### 8d  where(bothE.has) — EXISTS either direction

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(bothE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **where** | `EDGE_EXISTS bothE TRANSFER` |

**Results:** 10

,accountId,bankId
0,[801009860],[01674]
1,[800054B70],[012]
2,[800056160],[001]
3,[100428660],[070]
4,[800117590],[012]
5,[800115860],[00220]
6,[800119810],[010]
7,[800137450],[012]
8,[800164250],[010]
9,[80016DA30],[010]


In [45]:
show("g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)", title='8e  where(out.has) — EXISTS with vertex JOIN')


### 8e  where(out.has) — EXISTS with vertex JOIN

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `riskScore` ← `EDGE_PROPERTY: riskScore` |
| **where** | `OUT_NEIGHBOR_HAS out SENT_VIA` |

**Results:** 10

,accountId,bankId,riskScore
0,[8000F5030],[012],[0.58]
1,[8000F4FE0],[001],[0.21]
2,[812ED62E0],[0245335],[0.8]
3,[80012FD90],[010],[0.37]
4,[80012FE00],[012],[0.78]
5,[80012FE50],[001],[0.58]
6,[8005E18F0],[01665],[0.18]
7,[8005E24C0],[01665],[0.67]
8,[800131B10],[012],[0.54]
9,[8131A9A80],[0011642],[0.32]


In [46]:
show("g.V().hasLabel('Account').project('accountId','suspOut').by('accountId').by(outE('TRANSFER').has('isLaundering','1').count()).where(select('suspOut').is(gt(0))).order().by(select('suspOut'), Order.desc).limit(10)", title='8f  where(select.is(gt)) — HAVING-style filter')


### 8f  where(select.is(gt)) — HAVING-style filter

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','suspOut').by('accountId').by(outE('TRANSFER').has('isLaundering','1').count()).where(select('suspOut').is(gt(0))).order().by(select('suspOut'), Order.desc).limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **orderBy** | `suspOut` |
| **orderDirection** | `DESC` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `suspOut` ← `EDGE_DEGREE: out:TRANSFER:isLaundering=1` |
| **where** | `PROJECT_GT suspOut 0` |

**Results:** 10

,accountId,suspOut
0,[100428660],[33]
1,[1004286A8],[19]
2,[808338D40],[16]
3,[800737690],[16]
4,[80F984C70],[14]
5,[808E44B10],[13]
6,[8095AF170],[13]
7,[80F1AF380],[12]
8,[8001BB380],[10]
9,[1004286F0],[8]


---
## 9 — Compound `where`: `and`, `or`, `not`


In [47]:
show("g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY'))).project('accountId','bankId').by('accountId').by('bankId').limit(10)", title='9a  where(and(p1,p2)) — suspicious AND flagged')


### 9a  where(and(p1,p2)) — suspicious AND flagged

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY'))).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **where** | `AND  ` |

**Results:** 10

,accountId,bankId
0,[801009860],[01674]
1,[800054B70],[012]
2,[800056160],[001]
3,[100428660],[070]
4,[800117590],[012]
5,[800119810],[010]
6,[800137450],[012]
7,[8001A6140],[010]
8,[8001BB380],[010]
9,[8002B9DE0],[01420]


In [48]:
show("g.V().hasLabel('Account').where(or(out('SENT_VIA').has('fatfBlacklist','true'), outE('TRANSFER').has('isLaundering','1'))).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)", title='9b  where(or(p1,p2)) — blacklisted country OR suspicious')


### 9b  where(or(p1,p2)) — blacklisted country OR suspicious

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(or(out('SENT_VIA').has('fatfBlacklist','true'), outE('TRANSFER').has('isLaundering','1'))).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `riskScore` ← `EDGE_PROPERTY: riskScore` |
| **where** | `OR  ` |

**Results:** 10

,accountId,bankId,riskScore
0,[8000F5030],[012],[0.58]
1,[8000F4FE0],[001],[0.21]
2,[812ED62E0],[0245335],[0.8]
3,[80012FD90],[010],[0.37]
4,[80012FE00],[012],[0.78]
5,[80012FE50],[001],[0.58]
6,[8005E18F0],[01665],[0.18]
7,[8005E24C0],[01665],[0.67]
8,[800131B10],[012],[0.54]
9,[8131A9A80],[0011642],[0.32]


In [49]:
show("g.V().hasLabel('Account').where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId').by('accountId').by('bankId').limit(10)", title='9c  where(outE.count().is(0)) — NO alert flags')


### 9c  where(outE.count().is(0)) — NO alert flags

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **where** | `EDGE_EXISTS outE FLAGGED_BY` |

**Results:** 10

,accountId,bankId
0,[8000EBD30],[010]
1,[8000F4580],[03208]
2,[8000F5340],[001]
3,[8000F4670],[03209]
4,[8000F5030],[012]
5,[8000F5200],[010]
6,[8000F5AD0],[001]
7,[8000EBAC0],[001]
8,[8000EC1E0],[001]
9,[8000EC280],[012]


In [50]:
q9d_label, q9d, q9d_count = pick_first_non_empty([
    ('suspicious but unflagged',
     "g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId').by('accountId').by('bankId').limit(10)"),
    ('high-risk and unflagged',
     "g.V().hasLabel('Account').has('riskScore', gt(0.7)).where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId').by('accountId').by('bankId').limit(10)"),
    ('active and unflagged',
     "g.V().hasLabel('Account').where(and(outE('TRANSFER').count().is(gt(0)), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId').by('accountId').by('bankId').limit(10)"),
])
show(q9d, title=f'9d  where(and(p1,p2)) — {q9d_label} (count={q9d_count})')


### 9d  where(and(p1,p2)) — high-risk and unflagged (count=10)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('riskScore', gt(0.7)).where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `10` |
| **filter** | `riskScore > 0.7` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **where** | `EDGE_EXISTS outE FLAGGED_BY` |

**Results:** 10

,accountId,bankId
0,[8000F4580],[03208]
1,[8017BF800],[002439]
2,[80AEF5310],[0211050]
3,[8000F4510],[001]
4,[812ED62E0],[0245335]
5,[80012FE00],[012]
6,[8005F0B50],[010]
7,[8006A3840],[01420]
8,[8006AF500],[010]
9,[800AC9010],[021611]


---
## 10 — `groupCount` with alias key


In [51]:
show("g.V().hasLabel('Account').as('a').out('BELONGS_TO').as('b').groupCount().by(select('a').by('bankId')).order().by(values, desc).limit(10)", title='10a  groupCount().by(select(a).by(p)) — accounts per bank')


### 10a  groupCount().by(select(a).by(p)) — accounts per bank

**Gremlin:**
```groovy
g.V().hasLabel('Account').as('a').out('BELONGS_TO').as('b').groupCount().by(select('a').by('bankId')).order().by(values, desc).limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `groupCount` |
| **limit** | `10` |
| **orderDirection** | `DESC` |
| **hop[0]** | `out → ['BELONGS_TO']` via `iceberg.aml.account_bank` → `iceberg.aml.banks` |

**Results:** 10

,bankId,count
0,[012],[2499]
1,[010],[2092]
2,[001],[2082]
3,[003],[1704]
4,[015],[1583]
5,[007],[1583]
6,[020],[1546]
7,[028],[1476]
8,[0211],[1432]
9,[0116],[1430]


---
## 11 — Edge-root projections with `outV` / `inV`


In [52]:
show("g.E().hasLabel('TRANSFER').has('isLaundering','1').project('fromAcct','toAcct','amount','currency').by(outV().values('accountId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)", title='11a  g.E().project(outV.values, inV.values) — suspicious transfer endpoints')


### 11a  g.E().project(outV.values, inV.values) — suspicious transfer endpoints

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').has('isLaundering','1').project('fromAcct','toAcct','amount','currency').by(outV().values('accountId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `edge` |
| **rootLabel** | `TRANSFER` |
| **rootTable** | `iceberg.aml.transfers` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `15` |
| **filter** | `isLaundering = 1` |
| **project** | `fromAcct` ← `OUT_VERTEX_PROPERTY: accountId` |
| **project** | `toAcct` ← `IN_VERTEX_PROPERTY: accountId` |
| **project** | `amount` ← `EDGE_PROPERTY: amount` |
| **project** | `currency` ← `EDGE_PROPERTY: currency` |

**Results:** 15

,fromAcct,toAcct,amount,currency
0,[1004287C8],[8062015D0],[15027781.94],[Ruble]
1,[100428660],[800825340],[389769.39],[US Dollar]
2,[814009C90],[814009CE0],[3629.16],[US Dollar]
3,[803D3C380],[803D3CB00],[876.99],[US Dollar]
4,[8116734B0],[811673500],[3078.55],[Euro]
5,[805CAD3F0],[805CAD530],[148821.3],[Rupee]
6,[8044F5A10],[8044F5E00],[64469.82],[Yen]
7,[811B83280],[811C597B0],[13148.66],[Saudi Riyal]
8,[811C599A0],[811C597B0],[24189.19],[Saudi Riyal]
9,[100428660],[80365D680],[106.36],[US Dollar]


---
## 12 — `simplePath` across multi-hop traversals


In [53]:
show("g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)", title='12a  simplePath() — 3-hop money trail')


### 12a  simplePath() — 3-hop money trail

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **simplePath** | `True` |
| **limit** | `10` |
| **preHopLimit** | `1` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[1]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[2]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **pathBy** | `accountId` |
| **where** | `EDGE_EXISTS outE TRANSFER` |

**Results:** 0

---
## 13 — `identity()` as no-op and in `project`


In [54]:
show("g.V().hasLabel('Account').identity().project('accountId','bankId').by('accountId').by('bankId').limit(5)", title='13a  identity() as no-op')


### 13a  identity() as no-op

**Gremlin:**
```groovy
g.V().hasLabel('Account').identity().project('accountId','bankId').by('accountId').by('bankId').limit(5)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 5

,accountId,bankId
0,[8000EBD30],[010]
1,[8000F4580],[03208]
2,[8000F5340],[001]
3,[8000F4670],[03209]
4,[8000F5030],[012]


In [55]:
show("g.V().hasLabel('Account').project('graphId','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(5)", title='13b  project().by(identity()) — emit graph element ID')


### 13b  project().by(identity()) — emit graph element ID

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('graphId','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(5)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `5` |
| **project** | `graphId` ← `IDENTITY: id` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 5

,graphId,accountId,bankId
0,[1],[8000EBD30],[010]
1,[2],[8000F4580],[03208]
2,[3],[8000F5340],[001]
3,[4],[8000F4670],[03209]
4,[5],[8000F5030],[012]


---
## 14 — Full AML showcase query


In [56]:
q14_label, q14_query, q14_count = pick_first_non_empty([
    ('suspicious-but-unflagged', "g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId','riskScore','suspiciousOut','totalOut').by('accountId').by('bankId').by('riskScore').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).order().by(select('suspiciousOut'), Order.desc).limit(20)"),
    ('high-risk-and-unflagged',  "g.V().hasLabel('Account').has('riskScore', gt(0.7)).where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId','riskScore','suspiciousOut','totalOut').by('accountId').by('bankId').by('riskScore').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).order().by(select('suspiciousOut'), Order.desc).limit(20)"),
    ('active-and-unflagged',     "g.V().hasLabel('Account').where(and(outE('TRANSFER').count().is(gt(0)), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId','riskScore','suspiciousOut','totalOut').by('accountId').by('bankId').by('riskScore').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).order().by(select('suspiciousOut'), Order.desc).limit(20)"),
])
show(q14_query, title=f'14  Full AML showcase — {q14_label} (count={q14_count})', limit=20)


### 14  Full AML showcase — high-risk-and-unflagged (count=20)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('riskScore', gt(0.7)).where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId','riskScore','suspiciousOut','totalOut').by('accountId').by('bankId').by('riskScore').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).order().by(select('suspiciousOut'), Order.desc).limit(20)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **preHopLimit** | `20` |
| **orderBy** | `suspiciousOut` |
| **orderDirection** | `DESC` |
| **filter** | `riskScore > 0.7` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |
| **project** | `riskScore` ← `EDGE_PROPERTY: riskScore` |
| **project** | `suspiciousOut` ← `EDGE_DEGREE: out:TRANSFER:isLaundering=1` |
| **project** | `totalOut` ← `EDGE_DEGREE: out:TRANSFER` |
| **where** | `EDGE_EXISTS outE FLAGGED_BY` |

**Results:** 20

,accountId,bankId,riskScore,suspiciousOut,totalOut
0,[812ED62E0],[0245335],[0.8],[0],[6]
1,[80012FE00],[012],[0.78],[0],[6]
2,[8005F0B50],[010],[0.91],[0],[3]
3,[8006A3840],[01420],[0.76],[0],[7]
4,[8006AF500],[010],[0.91],[0],[2]
5,[800AC9010],[021611],[0.76],[0],[2]
6,[814233CC0],[001124],[0.74],[0],[2]
7,[800AC9DD0],[01601],[0.8],[0],[5]
8,[80152E780],[01665],[0.77],[0],[1]
9,[8016B82E0],[011296],[0.84],[0],[2]


---
## 15 — Deep multi-hop traversals (up to 10 hops)

> **How deep traversals execute:**
>
> | Provider | How hops run | SQL shown by `/query/explain` |
> |----------|-------------|-------------------------------|
> | `sql` (default, WCOJ on) | 1) Seed SQL for start vertices  2) **In-memory Leapfrog Trie Join** (no JOINs)  3) Property-fetch SQL | *Equivalent* JOIN plan — not what runs |
> | `sql` (WCOJ disabled) | Gremlin → N-JOIN SQL → JDBC | Exact SQL executed |
>
> WCOJ is **enabled by default** within the `sql` provider. It intercepts multi-hop
> path traversals and replaces the multi-JOIN SQL step with an in-memory
> sort-merge adjacency scan, which is 10–100× faster on dense graphs.
> The SQL shown for hop queries is a **reference plan** — it shows what the SQL
> engine *would* do without WCOJ.
> To disable: `WCOJ_ENABLED=false mvn exec:java`
>
> **To unlock 10-hop simple-path results**, reload with a larger slice:
> ```python
> MAX_ROWS = 500_000   # set at the top of Step 0, then re-run Step 2
> ```


In [57]:
# ── Data-size probe ─────────────────────────────────────────────────────────
_acct_count = _count("g.V().hasLabel('Account').count()")
_txn_count  = _count("g.E().hasLabel('TRANSFER').count()")
_LARGE_DATA = _acct_count >= 50_000   # True when ≥500 k rows loaded
_PROVIDER   = _active_provider()

display(Markdown(
    f'**Active provider:** `{_PROVIDER}` | '
    f'**Graph size:** {_acct_count:,} accounts, {_txn_count:,} transfers — '
    f'{"✅ large dataset (10-hop simple paths possible)" if _LARGE_DATA else "⚠️ small dataset (≤100 k rows) — 10-hop *simple* paths will return 0; set MAX_ROWS=500_000 in Step 0 and reload"}'
))
if _PROVIDER == 'wcoj':
    display(Markdown(
        '> ℹ️ **WCOJ mode** — each cell shows the **query plan** (resolved mapping decisions) '
        'instead of raw SQL. The equivalent SQL JOIN plan is collapsed under a disclosure triangle. '
        'Raw SQL is always in the server logs (`SQL_TRACE` lines).'
    ))

# Known laundering hub in the full HI-Small dataset.
# With ≥500 k rows this account (bank 100428660) has 14 k+ unique neighbors
# and a confirmed 10-hop simple-path chain.
_HUB = '100428660'


**Active provider:** `sql-multi` | **Graph size:** 423,684 accounts, 1,000,000 transfers — ✅ large dataset (10-hop simple paths possible)

### 15a — 10-hop simple money trail from laundering hub
The engine translates this to 10 consecutive JOINs with `NOT IN` cycle guards.
Each hop doubles the SQL complexity; this showcases the engine's ability to
produce arbitrarily-deep equi-join plans.


In [58]:
_q15a = (
    f"g.V().hasLabel('Account').has('accountId','{_HUB}')"
    f".repeat(out('TRANSFER').simplePath()).times(10)"
    f".path().by('accountId').limit(10)"
)
show(_q15a, title='15a  10-hop simplePath() money trail from laundering hub')


### 15a  10-hop simplePath() money trail from laundering hub

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('accountId','100428660').repeat(out('TRANSFER').simplePath()).times(10).path().by('accountId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **simplePath** | `True` |
| **limit** | `10` |
| **filter** | `accountId = 100428660` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[1]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[2]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[3]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[4]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[5]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[6]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[7]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[8]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[9]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **pathBy** | `accountId` |

**Results:** 10

1. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
2. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
3. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
4. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
5. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
6. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
7. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0

### 15b — 5-hop simple trail (works with 100 k rows too)
A shallower version that produces non-zero results even on the small demo
dataset, as long as at least one laundering account exists with ≥3 out-edges.


In [59]:
_q15b_label, _q15b, _q15b_count = pick_first_non_empty([
    ('5-hop from laundering hub',
     f"g.V().hasLabel('Account').has('accountId','{_HUB}')"
     f".repeat(out('TRANSFER').simplePath()).times(5)"
     f".path().by('accountId').limit(10)"),
    ('5-hop from any laundering seed',
     "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1)"
     ".repeat(out('TRANSFER').simplePath()).times(5)"
     ".path().by('accountId').limit(10)"),
    ('4-hop from any laundering seed',
     "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1)"
     ".repeat(out('TRANSFER').simplePath()).times(4)"
     ".path().by('accountId').limit(10)"),
    ('3-hop from any laundering seed',
     "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1)"
     ".repeat(out('TRANSFER').simplePath()).times(3)"
     ".path().by('accountId').limit(10)"),
    ('2-hop from any laundering seed',
     "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1)"
     ".repeat(out('TRANSFER').simplePath()).times(2)"
     ".path().by('accountId').limit(10)"),
])
show(_q15b, title=f'15b  Deep simplePath() — {_q15b_label} (count={_q15b_count})')


### 15b  Deep simplePath() — 5-hop from laundering hub (count=10)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('accountId','100428660').repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **simplePath** | `True` |
| **limit** | `10` |
| **filter** | `accountId = 100428660` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[1]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[2]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[3]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[4]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **pathBy** | `accountId` |

**Results:** 10

1. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
2. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
3. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
4. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
5. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
6. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
7. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
8. ['100428660', '800320550', '800437170', '800E93380', '808224AD0', '8082250C0']
9. ['100428660', '80022A3F0', '80025CBF0', '80032B020', '80152E240', '80A0DDC70']
10. ['100428660', '80022A3F0', '80025CBF0', '80032B020', '80152E240', '80A0DDC70']


### 15c — Count distinct accounts reachable in exactly 10 hops (no cycle guard)
Without `simplePath()` the engine allows revisiting nodes, so cycles are
traversed freely.  This turns into a pure 10-JOIN COUNT query — always returns
a non-zero number as long as the hub has at least one out-edge.


In [60]:
show(
    f"g.V().hasLabel('Account').has('accountId','{_HUB}')"
    f".repeat(out('TRANSFER')).times(10)"
    f".dedup().count()",
    title='15c  COUNT DISTINCT accounts reachable in 10 hops (cycles allowed)'
)


### 15c  COUNT DISTINCT accounts reachable in 10 hops (cycles allowed)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('accountId','100428660').repeat(out('TRANSFER')).times(10).dedup().count()
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **aggregation** | `count` |
| **dedup** | `True` |
| **filter** | `accountId = 100428660` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[1]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[2]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[3]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[4]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[5]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[6]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[7]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[8]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[9]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |

**Results:** 1

1. 38409


In [61]:
show(
    f"g.V().hasLabel('Account').has('accountId','{_HUB}')"
    f".repeat(out('TRANSFER')).times(5)"
    f".dedup().project('accountId','bankId').by('accountId').by('bankId').limit(20)",
    title='15d  Project 5-hop downstream accounts (cycles allowed, DISTINCT)'
)


### 15d  Project 5-hop downstream accounts (cycles allowed, DISTINCT)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('accountId','100428660').repeat(out('TRANSFER')).times(5).dedup().project('accountId','bankId').by('accountId').by('bankId').limit(20)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **limit** | `20` |
| **dedup** | `True` |
| **filter** | `accountId = 100428660` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[1]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[2]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[3]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[4]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **project** | `accountId` ← `EDGE_PROPERTY: accountId` |
| **project** | `bankId` ← `EDGE_PROPERTY: bankId` |

**Results:** 20

,accountId,bankId
0,[805B34210],[013037]
1,[8005F2D30],[01420]
2,[810B0FB40],[013327]
3,[8082250C0],[0111764]
4,[8065002B0],[015723]
5,[80974B620],[012]
6,[805DEB4F0],[023885]
7,[80A0DDC70],[001047]
8,[80D120A40],[0014804]
9,[8018859B0],[023833]


### 15e — 10-hop reachability from top-5 laundering accounts (groupCount)
Finds the five most-connected laundering accounts and counts how many
distinct nodes each can reach in exactly 10 hops.  Demonstrates that
deep traversals compose naturally with aggregation.


In [62]:
# Discover top laundering seeds dynamically
_seeds_res = run(
    "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1'))"
    ".project('accountId','outDeg').by('accountId').by(outE('TRANSFER').count())"
    ".order().by(select('outDeg'), Order.desc).limit(5)"
)
_seeds = [r['accountId'] for r in (_seeds_res.get('results') or []) if isinstance(r, dict)]
if not _seeds:
    _seeds = [_HUB]

display(Markdown(f'**Top laundering seeds found:** `{_seeds}`'))

# Run 10-hop reachability count for each seed
_reach_rows = []
for _s in _seeds:
    _cnt = _count(
        f"g.V().hasLabel('Account').has('accountId','{_s}')"
        f".repeat(out('TRANSFER')).times(10).dedup().count()"
    )
    _reach_rows.append({'seed_accountId': _s, '10hop_reachable_distinct': _cnt})

if _reach_rows:
    display(Markdown('**10-hop reachability per seed:**'))
    display(pd.DataFrame(_reach_rows))


**Top laundering seeds found:** `[['100428660'], ['1004286A8'], ['100428978'], ['1004286F0'], ['1004289C0']]`

**10-hop reachability per seed:**

,seed_accountId,10hop_reachable_distinct
0,[100428660],0
1,[1004286A8],0
2,[100428978],0
3,[1004286F0],0
4,[1004289C0],0


### 15f — Full 10-hop simple-path chain with edge amounts (large dataset only)
When loaded with ≥500 k rows, shows the complete chain of account IDs *and*
the transfer amounts along each hop — a realistic AML investigation query.


In [63]:
if _LARGE_DATA:
    show(
        f"g.V().hasLabel('Account').has('accountId','{_HUB}')"
        f".repeat(out('TRANSFER').simplePath()).times(10)"
        f".path().by('accountId').limit(5)",
        title='15f  Full 10-hop simplePath chain (large dataset)'
    )
else:
    display(Markdown(
        '### 15f  Full 10-hop simplePath chain\n'
        '⚠️ **Skipped** — requires `MAX_ROWS ≥ 500_000`.  '
        'Set `MAX_ROWS = 500_000` in **Step 0** and re-run **Step 2** to load more data.\n\n'
        'With the full dataset the expected SQL looks like:\n'
        '```sql\n'
        'WITH _start AS (\n'
        '  SELECT * FROM accounts WHERE account_id = ?\n'
        ')\n'
        'SELECT v0.account_id AS accountId0,\n'
        '       v1.account_id AS accountId1, ...,\n'
        '       v10.account_id AS accountId10\n'
        'FROM _start v0\n'
        'JOIN transfers e1 ON e1.out_id = v0.id JOIN accounts v1 ON v1.id = e1.in_id\n'
        'JOIN transfers e2 ON e2.out_id = v1.id JOIN accounts v2 ON v2.id = e2.in_id\n'
        '...  (10 JOIN pairs total)\n'
        'WHERE v1.id <> v0.id\n'
        '  AND v2.id NOT IN (v0.id, v1.id)\n'
        '  ...\n'
        '  AND v10.id NOT IN (v0.id, v1.id, ..., v9.id)\n'
        'LIMIT 5\n'
        '```'
    ))


### 15f  Full 10-hop simplePath chain (large dataset)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('accountId','100428660').repeat(out('TRANSFER').simplePath()).times(10).path().by('accountId').limit(5)
```

**Execution engine:** `SQL`

**Query plan:**

| Field | Value |
|-------|-------|
| **rootType** | `vertex` |
| **rootLabel** | `Account` |
| **rootTable** | `iceberg.aml.accounts` |
| **dialect** | `IcebergSqlDialect` |
| **simplePath** | `True` |
| **limit** | `5` |
| **filter** | `accountId = 100428660` |
| **hop[0]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[1]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[2]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[3]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[4]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[5]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[6]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[7]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[8]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **hop[9]** | `out → ['TRANSFER']` via `iceberg.aml.transfers` → `iceberg.aml.accounts` |
| **pathBy** | `accountId` |

**Results:** 5

1. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
2. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
3. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
4. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']
5. ['100428660', '800055DF0', '800111C30', '80011F1C0', '80014BF30', '800182360', '8004E6440', '80059CE00', '805B63CA0', '809F2A060', '8131A9A80']


---
## WCOJ Disk Index Status

Run this cell at any time to check whether WCOJ disk index files were written.

> **When are files written?**
> - WCOJ must be enabled (default: `WCOJ_ENABLED=true`)
> - A hop query (e.g. sections 4, 12, 15) must have been executed
> - The edge table must exceed `WCOJ_MAX_EDGES` (default 5 M rows → in-memory;
>   set `WCOJ_MAX_EDGES=500000` to force disk for a 1 M-row table)
>
> With the default settings and 1 M TRANSFER rows the index stays **in-memory**
> (faster). Set `WCOJ_MAX_EDGES=500000` when starting the server to force disk.
>
> **Index TTL:** By default indices are rebuilt after 30 seconds (`WCOJ_INDEX_TTL_SECONDS=30`).
> Set `WCOJ_INDEX_TTL_SECONDS=0` to disable TTL and rely on manual invalidation only.
> To force an immediate rebuild call `POST /wcoj/invalidate` or delete the cache directory.


In [64]:
import pathlib, os

# ── Locate repo root ──────────────────────────────────────────────────────────
_repo = next((p for p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents)
              if (p / 'pom.xml').exists()), pathlib.Path.cwd())
_cache_dir = _repo / 'wcoj-index-cache'

# ── Check active provider ─────────────────────────────────────────────────────
try:
    import requests as _req
    _prov = _req.get(f'{BASE_URL}/gremlin/provider', timeout=5).json().get('provider', '?')
except Exception:
    _prov = 'unknown (server not reachable)'

display(Markdown(f'**Active provider:** `{_prov}`'))

# WCOJ is now built into the sql provider — disk index is written whenever
# WCOJ_ENABLED=true (default) and a hop query runs with edge table > WCOJ_MAX_EDGES.
# No need to restart with a different GRAPH_PROVIDER.

# ── Scan cache directory ──────────────────────────────────────────────────────
if not _cache_dir.exists():
    display(Markdown(
        f'**Cache directory:** `{_cache_dir}` — **not found**\n\n'
        '> No disk index has been written yet. '
        'Run a hop query (sections 4, 12, or 15) while the server is in `wcoj` mode.'
    ))
else:
    _rows = []
    _total = 0
    for _mapping_dir in sorted(_cache_dir.iterdir()):
        if not _mapping_dir.is_dir(): continue
        for _f in sorted(_mapping_dir.iterdir()):
            if _f.suffix not in ('.csr',): continue
            _sz = _f.stat().st_size
            _total += _sz
            _direction = 'out' if _f.stem.endswith('.out') else 'in'
            _rows.append({
                'mapping':   _mapping_dir.name,
                'file':      _f.name,
                'direction': _direction,
                'size':      f'{_sz / 1024 / 1024:.1f} MB',
                'modified':  _f.stat().st_mtime,
            })

    if not _rows:
        display(Markdown(
            f'**Cache directory exists** (`{_cache_dir}`) but **no `.csr` files found**.\n\n'
            '> Run a hop query while the server is in `wcoj` mode.'
        ))
    else:
        import pandas as _pd, datetime as _dt
        _df = _pd.DataFrame(_rows)
        _df['modified'] = _df['modified'].apply(
            lambda ts: _dt.datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M:%S')
        )
        display(Markdown(
            f'✅ **WCOJ disk index found** — `{len(_rows)}` file(s), '
            f'total `{_total / 1024 / 1024:.1f} MB`\n\n'
            f'**Cache root:** `{_cache_dir}`'
        ))
        display(_df.drop(columns=['modified']).assign(modified=_df['modified']))
        display(Markdown(
            '> **Files persist across server restarts** — '
            'WCOJ reuses them on next startup. '
            'To force rebuild: call `POST /wcoj/invalidate` or delete the cache directory.\n\n'
            '> **Quota:** `WCOJ_DISK_QUOTA_MB` env var (default 2048 MB per mapping). '
            'If exceeded, WCOJ falls back to SQL for that edge table.'
        ))


**Active provider:** `sql-multi`

> ℹ️ WCOJ is built into the `sql` provider and enabled by default (`WCOJ_ENABLED=true`).
> Disk index files are written here when a hop query runs with edge table > `WCOJ_MAX_EDGES`.
> To force disk index for 1 M rows: `WCOJ_MAX_EDGES=500000 mvn exec:java`
> To disable WCOJ entirely: `WCOJ_ENABLED=false mvn exec:java`

✅ **WCOJ disk index found** — `2` file(s), total `27.4 MB`

**Cache root:** `/Users/vjoshi/SourceCode/graph-query-engine/wcoj-index-cache`

,mapping,file,direction,size,modified
0,aml-iceberg,iceberg_aml_transfers_out_id_in_id.in.csr,in,13.6 MB,2026-04-10 17:36:05
1,aml-iceberg,iceberg_aml_transfers_out_id_in_id.out.csr,out,13.8 MB,2026-04-10 17:36:05


> **Files persist across server restarts** — WCOJ reuses them on next startup. To force rebuild: call `POST /wcoj/invalidate` or delete the cache directory.

> **Quota:** `WCOJ_DISK_QUOTA_MB` env var (default 2048 MB per mapping). If exceeded, WCOJ falls back to SQL for that edge table.

---
## Summary

| Section | Steps demonstrated |
|---------|--------------------|
| 1 | `g.V()`, `g.E()`, `g.V(id)` — root steps |
| 2 | `has`, `hasLabel`, `hasId`, `hasNot`, `is(pred)` — filter steps |
| 3 | `out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV` — hop steps |
| 4 | `repeat().times(n)`, `simplePath()`, `path().by(p)` — path steps |
| 5 | `count`, `sum`, `mean`, `groupCount().by(p)`, `dedup` — aggregation |
| 6 | `values`, `project().by()`, `valueMap`, `by(identity())`, `by(out.values.fold)`, `by(outE/inE.count)`, `by(choose)`, `by(outV/inV.values)` — projection |
| 7 | `order().by(p)`, `order().by(select(a))`, `limit(n)` — ordering/paging |
| 8 | `as`, `select().by`, `where(neq)`, `where(outE/inE/bothE.has)`, `where(out.has)`, `where(select.is(gt))` — alias/where |
| 9 | `where(and(...))`, `where(or(...))`, `where(not(...))` — compound predicates |
| 10 | `groupCount().by(select(a).by(p))` — alias-keyed grouping |
| 11 | `g.E().project(by(outV.values), by(inV.values))` — edge-root projections |
| 12 | `simplePath()` across multi-hop `repeat` |
| 13 | `identity()` as no-op and in `project().by(identity())` |
| 14 | Full combined query |
| 15 | `repeat().times(10)` — 10-hop simple & non-simple deep traversals, reachability counts |

> Switch `BACKEND = 'iceberg'` in **Step 0** and re-run from the top to compare SQL dialects.
